# Demand Forecasting → Service-Constrained Inventory Decisions

**M5 Forecasting — Accuracy · Retail planning research · 28-day demand horizon**

> **Decision question:** Does a demand forecast produce a lower-cost replenishment plan **without hurting product availability**?  A smaller forecast error by itself is not sufficient.

**What → Why → How → Outcome:** Train leakage-safe recursive forecasts → prevent excess holding cost and stockouts → tune demand forecasts + lead-time safety stock on a separate 28-day period → judge the frozen strategy on an untouched 28-day test; report **pass or fail**, not just the cheapest result.

**Important:** Retailer stock balances, replenishment lead times, purchase orders, unit carrying costs, and actual lost demand are **not provided by M5**. Every stock/cost figure below is synthetic and meant to illustrate decision quality, not savings at any business.

## 01 / Configuration

Download the Kaggle [M5 Forecasting — Accuracy](https://www.kaggle.com/competitions/m5-forecasting-accuracy) files and place `calendar.csv`, `sell_prices.csv`, and `sales_train_validation.csv` in `data/m5`. Run from the project root after `pip install -r requirements.txt`.

Set `min_fill_rate=0.96` as a **hypothetical service objective**, not a standard derived from the dataset. The final decision also requires a fill rate **at least as high as the test-period fixed baseline**.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from analysis.pipeline import Config, load_m5, chronological_splits, run

DATA_DIR = Path('data/m5')
OUTPUT_DIR = Path('dashboard/public/data')
cfg = Config(data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
             top_n=75, horizon=28, min_fill_rate=0.96, random_state=42)
print('Data folder:', DATA_DIR.resolve())

## 02 / Audit dataset coverage and split boundaries

Select the most active items using **fit-history sales only**. The last 56 days are split into a 28-day tuning period and a 28-day **untouched test period**. Price features for future dates are held at the latest pre-cutoff observed price; calendar and SNAP signals are treated as known in advance.

In [ ]:
demand, calendar, _ = load_m5(cfg)
fit, tune, test = chronological_splits(demand, cfg.horizon)
segments = pd.DataFrame([
    {'segment': name, 'rows': len(block),
     'start': block.date.min().date(), 'end': block.date.max().date()}
    for name, block in [('Fit history',fit),('Tune model + policy',tune),('Untouched test',test)]
])
display(segments)
print('SKUs:', demand.item_id.nunique(),
      '| SKU-store pairs:', demand[['item_id','store_id']].drop_duplicates().shape[0])

## 03 / Forecasting methodology

- Inputs: shifted 7/14/28-day lags, shifted 7/28-day means, seasonal/calendar indicators, SNAP, price and event signals.
- Candidates: seven-day moving mean, linear regression, random forest, XGBoost.
- Validation: fully **recursive** 28-day walk-forward predictions. From day two onward, use earlier **predictions**, never realized holdout sales, as lag input.
- Select the model using **tuning WAPE only**; refit on fit + tuning data, then evaluate test WAPE without changing the champion.

This deliberately differs from the uploaded notebook's earlier rolling-lag WAPE, which accessed actuals inside the evaluation window.

## 04 / Operational model and constrained optimization

For each synthetic item/store: lead time is 3/5/7 days, simulated on-hand inventory is last seven fit-history sales, carrying penalty is `0.02` per unit/day, and stockout penalty is `1.50` per unfulfilled unit. These **arbitrary cost units are not dollars**.

At the end of each day, the forecast policy calculates:

\[
S_t = m\sum_{k=1}^{L+1}\widehat{d}_{t+k} + z\sigma\sqrt{L+1}
\]

Here `L` = lead time in days, `+1` = review interval, `m` = forecast calibration multiplier, `z` = safety-stock multiplier, `σ` = historical daily demand standard deviation. Reorder quantity is `max(0, Sₜ − on_hand − on_order)`. Today's demand is served *before* today's order is created; the order arrives after `L` days.

A fairer non-ML baseline uses historical mean × (`L+1`) plus one standard deviation of lead/review demand, and accounts for in-transit units too. It is not artificially starved of safety stock.

On tuning data only, test safety multipliers `0, 0.5, 1, 1.5, 2, 2.5, 3` and calibration multipliers `0.9, 1.0, 1.1`, plus the fixed baseline. Pick the **lowest-cost** strategy subject to `fill_rate ≥ 96%`; if none qualifies, select none. Because the forecast model and the stock policy are both selected on the tuning interval, the final test is crucial.

## 05 / Reproduce the study (single execution)

`run()` performs model selection and policy tuning **before** testing. It exports a full tuning candidate grid, a *diagnostic-only* test sweep (never used for selection), selected-vs-baseline test outcomes, a future forecast, and a dashboard JSON. Expect several minutes on full M5 depending on hardware.

In [ ]:
report = run(cfg)
print('Chosen model on tuning:', report['meta']['champion'])
print('Locked policy:', report['selected_policy'])
print('Final decision:', report['decision']['status'])
print('Why:', report['decision']['reason'])
display(pd.DataFrame(report['policies']))

## 06 / Inspect the holdout decision, not just model accuracy

**A proposed forecast policy is marked successful only if all three conditions hold on the untouched test:**

1. Fill rate ≥ configured minimum (96% in this study).
2. Fill rate ≥ test-period fixed baseline.
3. Modeled cost < test-period fixed baseline.

If the optimizer selects the baseline on tuning, if no policy is feasible, or if a challenger fails **any** of these tests, the final recommendation is to **not replace** the fixed baseline. This is valid, publishable negative evidence: do not rewrite the result to force ML to win.

In [ ]:
tuning = pd.read_csv(OUTPUT_DIR/'policy_tuning_grid.csv')
selected = pd.read_csv(OUTPUT_DIR/'policy_comparison.csv')
model_tune = pd.read_csv(OUTPUT_DIR/'model_comparison_tuning.csv')
model_test = pd.read_csv(OUTPUT_DIR/'model_comparison_test.csv')
print('Best tuning candidate vs frozen test outcome:')
display(tuning.sort_values('total_inventory_cost').head(8))
display(selected[['policy','fill_rate','total_inventory_cost','stockout_units','average_inventory']])
print('Model rankings for comparison (test rankings are descriptive only):')
display(model_tune.merge(model_test,on='model',suffixes=('_tuning','_test')))

## 07 / Inspect operational trade-offs

The following chart shows *tuning-period* cost versus service across tested safety-stock settings. Only candidates above the configured fill floor qualify for selection. It does **not** depict measured retailer savings.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,5))
for name, group in tuning.groupby('strategy'):
    ax.scatter(group.fill_rate*100, group.total_inventory_cost,
               label=name, alpha=.75, s=45)
ax.axvline(cfg.min_fill_rate*100, ls='--', label='Required fill rate')
ax.set(xlabel='Tuning demand fill rate (%)',
       ylabel='Holding + stockout penalties (synthetic cost units)',
       title='Safety-stock tuning: cost versus service')
ax.legend()
plt.show()

## 08 / Operational recommendations and deliverables

`inventory_recommendations.csv` contains **illustrative** planned inventory position and order quantities for each SKU/store. For a deployed implementation this would require true on-hand, on-order, lead-time distributions, supplier order rules, and stockout-censored demand adjustments.

The interactive dashboard consumes `dashboard/public/data/results.json`. Use Vercel with `dashboard` as the root directory. The standalone `Dashboard_Preview.html` preserves the **archived** results and is intentionally not represented as a newly optimized outcome.

### Known limitations
- M5 is *units sold*, not guaranteed unconstrained demand; sales censored by stockouts may bias forecasts.
- Missing future realized prices are replaced with most recent pre-cutoff observed prices (a documented assumption).
- Replenishment parameters and costs are simulated; there are no real business savings.
- One two-window historical backtest may not generalize; next step is multi-origin seasonal stress testing.
- Modeling fractional inventory units is a simplifying choice; real ordering rounds to pack sizes.
- The simulated beginning stock is not verified on-hand stock; varying it is important sensitivity testing.
- The 2016 forecasts are historical, not live predictions.

**Do not copy archived 4.27% cost reduction into the revised study's outcome without rerunning M5.**